## Modelos de Machine Learning: XGBoost, Sistema de Predicción Energética en Barcelona


> Hermano del notebook 06 (LightGBM). El objetivo es el mismo: comprobar si los modelos de árboles
> capturan relaciones no lineales e interacciones, ahora con **XGBoost**.

### Principio de comparación justa

- el mismo split temporal (train 2019–2024 · validación ene–sep 2025 · test oct–nov 2025 intacto)
- las mismas métricas (R² primaria; MAE, RMSE, MAPE)
- los mismos horizontes de backtesting (24h, 48h, 72h)
- el mismo arnés (skforecast: ForecasterRecursiveMultiSeries + backtesting walk-forward, refit=False)

> La única diferencia de fondo frente al 06 es el regresor interno (`XGBRegressor` en vez de
> `LGBMRegressor`). Todo lo demás se mantiene para que la comparación XGBoost vs LightGBM dependa
> solo del algoritmo, no del montaje.

> **Alcance de este notebook:** llegamos hasta el **control de overfitting del modelo base**
> (regla José). A partir de ahí se decide qué hacer con el modelo (tuning, poda de variables, etc.).

## Librerías

In [ ]:
import os
os.environ['TQDM_DISABLE'] = '1'   # mata barras tqdm/ipywidgets (bug de render de VS Code)
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pymongo import MongoClient
import warnings, time
import skforecast; print(skforecast.__version__)
from xgboost import XGBRegressor        # <- unico cambio de fondo frente al 06 (LGBMRegressor)
import shap
from skforecast.recursive import ForecasterRecursiveMultiSeries, ForecasterEquivalentDate
from skforecast.preprocessing import RollingFeatures
from skforecast.model_selection import (
    TimeSeriesFold,
    backtesting_forecaster_multiseries,
)
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


warnings.filterwarnings('ignore')

plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#D3D3D3'
plt.rcParams['grid.linewidth'] = 0.4
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Paleta
C1 = '#264653'; C2 = '#2A9D8F'; C3 = '#E9C46A'
C4 = '#F4A261'; C5 = '#E76F51'; C6 = '#A8DADC'
TITULO = '#1B3A5C'; SUBTITULO = '#C0392B'

start_time = time.time()

---
## Carga de datos

In [ ]:
client = MongoClient('mongodb://mongo:27017/')
db     = client['tfm_energy']

docs = list(db['dataset_ml'].find({}, {'_id': 0}))
df_ml   = pl.DataFrame(docs, infer_schema_length=None)

print(f"Shape: {df_ml.shape}")
print(f"Desde: {df_ml['datetime'].min()}  Hasta: {df_ml['datetime'].max()}")
print(f"Codigos postales: {df_ml['cod_postal'].n_unique()}")
print(f"Columnas ({len(df_ml.columns)}):")
print(sorted(df_ml.columns))

---
# <font color='#1B3A5C'>  **1. Configuración del Experimento** </font>

> mismos cortes temporales y horizontes que SARIMA/SARIMAX (05) y LightGBM (06) para que la
> comparación baseline vs. ML sea justa. El conjunto de *test* (oct–nov 2025) queda intacto.

In [ ]:
# Identicos a SARIMA, SARIMAX y LightGBM (06)
INI       = '2019-01-01'             # historico completo
FIN_TRAIN = '2024-12-31 18:00:00'    # fin de train
FIN_VAL   = '2025-09-30 18:00:00'    # fin de validacion (2025-01 -> 2025-09)
# test: 2025-10-01 a  2025-11-30 (lo que queda tras FIN_VAL), INTACTO

#  4=24h, 8=48h, 12=72h
HORIZONTES = {'24h': 4, '48h': 8, '72h': 12}

TARGET = 'mwh_total'
# SOLO LIMPIOS (auditoria de calidad de datos, notebook 02): 12 CPs con target corrupto
# en val/test por reasignacion de OpenData -> excluidos. Quedan 30 limpios.
CPS_SUCIOS = ['08011','08009','08007','08013','08010','08006','08005',
              '08019','08008','08036','08026','08037']
CPS_TODOS = [cp for cp in sorted(df_ml['cod_postal'].unique().to_list()) if cp not in CPS_SUCIOS]

print(f"Train : {INI} -> {FIN_TRAIN}")
print(f"Val   : 2025-01-01 -> {FIN_VAL}")
print(f"Test  : 2025-10-01 -> {df_ml['datetime'].max()}  (intacto)")
print(f"Horizontes: {HORIZONTES}")
print(f"Target: {TARGET} | CPs: {len(CPS_TODOS)}")

### <font color='#C0392B'><b>1.1 Los cuatro roles de las variables</b></font>

> Hay que asignar a cada variable su papel para skforecast.

 - Identificadores datetime, cod_postal estructuran serie y tiempo.
 - Target mwh_total.
 - Autorregresivas lags, rollings, se descartan; skforecast las regenera sin leakage.
 - Exógenas como clima, calendario y perfil sectorial los regresores externos que sí entran.

In [ ]:
LAGS = [1, 2, 3, 4, 28]
AUTOREG = ['lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_28', 'rolling_mean_7d', 'rolling_std_7d']
NO_EXOG = ['datetime', 'cod_postal', TARGET] + AUTOREG
EXOG = [c for c in df_ml.columns if c not in NO_EXOG]

### <font color='#C0392B'><b>1.2 Construcción de las series multi-series</b></font>

> ForecasterRecursiveMultiSeries no recibe la tabla plana, sino las 30 series agrupadas por código postal.

 - series y exog serán un diccionario por cada cosa.
 - Cada serie se reindexa a frecuencia de 6h (skforecast lo exige).
 - NaN residuales se mantienen, XGBoost los gestiona de forma nativa (igual que LightGBM), ventaja frente a SARIMAX.

In [ ]:
series = {}   # {cod_postal: pd.Series  (mwh_total, index 6h)}
exog   = {}   # {cod_postal: pd.DataFrame (EXOG, index 6h)}

for cp in CPS_TODOS:
    # Solo filas de ese barrio a pandas, la fecha es indice y marca frecuencia a 6h
    g = (df_ml.filter(pl.col('cod_postal') == cp)
              .sort('datetime')
              .to_pandas()
              .set_index('datetime'))
    g.index = pd.DatetimeIndex(g.index)
    g = g.asfreq('6h')

    # sanear exogenas booleanas para que funcione bien en los modelos de ML
    ex = g[EXOG].copy()
    bool_cols = ex.select_dtypes('bool').columns
    ex[bool_cols] = ex[bool_cols].astype('int8')

    series[cp] = g[TARGET]
    exog[cp]   = ex

# Comprobacion rapida
cp0 = CPS_TODOS[0]
print(f"Series: {len(series)} CPs")
print(f"{cp0}: {len(series[cp0])} bloques | exog {exog[cp0].shape} | "
      f"NaN exog: {int(exog[cp0].isna().sum().sum())}")
print("Columnas exog:", list(exog[cp0].columns))

print(series[cp0].head())
print(exog[cp0].head())

> El resultado son dos diccionarios con los CP de key y sus respectivas variables de value.

---
# <font color='#1B3A5C'>  **2. Modelo base: XGBoost multi-series** </font>

> Primer modelo con hiperparámetros por defecto, para tener una referencia antes de optimizar.
- Un único modelo que aprende de los 30 CPs a la vez.
- RollingFeatures genera el rolling_mean y std de 7 días del feature engineering.

In [ ]:
rolling = RollingFeatures(stats=['mean', 'std'], window_sizes=28)   # 28 bloques = 7 dias

# OJO XGBoost: necesita enable_categorical=True para digerir la columna de nivel (cod_postal),
# que skforecast crea como 'category' con encoding='ordinal_category'. LightGBM lo hacia solo;
# es el unico ajuste de API frente al 06. (Si diera problemas, alternativa: encoding='ordinal'.)
forecaster = ForecasterRecursiveMultiSeries(
    estimator       = XGBRegressor(random_state=12, verbosity=0, enable_categorical=True),
    lags            = LAGS,
    window_features = rolling,
    encoding        = 'ordinal_category',
)
forecaster

In [ ]:
def metricas(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    denom = np.where(y_true == 0, np.nan, y_true)
    return {
        'r2'  : r2_score(y_true, y_pred),
        'mae' : mean_absolute_error(y_true, y_pred),
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mape': float(np.nanmean(np.abs((y_true - y_pred) / denom)) * 100),
    }

### <font color='#C0392B'><b>2.1 Entrenamiento y backtest base</b></font>

In [ ]:
series_val = {cp: s.loc[:FIN_VAL] for cp, s in series.items()}
exog_val   = {cp: e.loc[:FIN_VAL] for cp, e in exog.items()}

fin_train_idx = len(series[CPS_TODOS[0]].loc[:FIN_TRAIN])

cv_val = TimeSeriesFold(steps=12, initial_train_size=fin_train_idx, refit=False)


metrica_sk, predicciones = backtesting_forecaster_multiseries(
    forecaster = forecaster,
    series     = series_val,
    exog       = exog_val,
    cv         = cv_val,
    metric     = 'mean_absolute_error',   # obligatorio pasar una; la ignoraremos
    add_aggregated_metric = False,
    verbose    = False,
    show_progress = False,
)


print("predicciones:", predicciones.shape)
print(predicciones.head())

### <font color="#C0392B"><b>2.2 Revisión de resultados (función reutilizable)</b></font>

> `revisar_resultados()` reúne en una sola llamada toda la diagnosis del modelo desde su backtest:
> métricas por CP (R² primaria), R² por barrio, predicho vs real, error por horizonte, error por
> hora/día, curva train vs val, importancia nativa y SHAP. Es la misma función del 06, adaptada a
> la API de XGBoost (curva de entrenamiento y SHAP).

In [ ]:
def revisar_resultados(predicciones, series_val, forecaster, exog_val,
                       nombre='XGBoost', excluir=(),
                       cps_demo=('08002', '08028', '08039'),
                       ventana=('2025-03-01', '2025-03-31'),
                       shap_on=True, shap_sample=3000):
    """Dashboard de revision de un modelo desde su backtest. Devuelve ml_val."""
    cps = sorted(predicciones['level'].unique())

    # ---------- calculos ----------
    # metricas por CP
    filas = []
    for cp in cps:
        p = predicciones.loc[predicciones['level'] == cp, 'pred'].sort_index()
        y = series_val[cp].reindex(p.index)
        m = metricas(y.values, p.values); m['cp'] = cp; filas.append(m)
    ml_val = pd.DataFrame(filas)
    val = ml_val[~ml_val['cp'].isin(excluir)]

    # error por horizonte
    err = []
    for cp in cps:
        if cp in excluir:
            continue
        sub = predicciones[predicciones['level'] == cp].sort_index().copy()
        sub['ae']   = np.abs(series_val[cp].reindex(sub.index).values - sub['pred'].values)
        sub['step'] = sub.groupby('fold').cumcount() + 1
        err.append(sub[['step', 'ae']])
    curva = pd.concat(err).groupby('step')['ae'].mean()

    # error por hora / dia
    tt = []
    for cp in cps:
        if cp in excluir:
            continue
        sub = predicciones[predicciones['level'] == cp].sort_index()
        tt.append(pd.DataFrame({'ae': np.abs(series_val[cp].reindex(sub.index).values - sub['pred'].values)}, index=sub.index))
    tt = pd.concat(tt); tt['hora'] = tt.index.hour; tt['dia'] = tt.index.dayofweek

    # curva train/val + importancia (reentrena un XGB con los params del forecaster)
    out = forecaster.create_train_X_y(series=series_val, exog=exog_val)
    X_all, y_all = out[0], out[1]
    # la columna de nivel viene como 'category'; para el XGB de diagnostico la pasamos a codigos
    # enteros (evita depender de enable_categorical en SHAP y en la curva). Es solo diagnostico.
    X_all = X_all.copy()
    for c in X_all.select_dtypes('category').columns:
        X_all[c] = X_all[c].cat.codes
    mtr = X_all.index <= FIN_TRAIN
    X_tr, y_tr, X_va, y_va = X_all[mtr], y_all[mtr], X_all[~mtr], y_all[~mtr]
    reg_base = getattr(forecaster, 'regressor', None) or getattr(forecaster, 'estimator', None)
    params = reg_base.get_params(); params['eval_metric'] = 'rmse'
    reg = XGBRegressor(**params)
    reg.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_va, y_va)], verbose=False)
    ev_raw = reg.evals_result()                       # XGB: dict con validation_0 (train) y validation_1 (val)
    ev = {'train': {'rmse': ev_raw['validation_0']['rmse']},
          'val':   {'rmse': ev_raw['validation_1']['rmse']}}
    imp = pd.Series(reg.feature_importances_, index=X_tr.columns).sort_values(ascending=False)

    # ---------- resumen numerico ----------
    print(f"=== {nombre} - validacion, {len(val)} CPs limpios ===")
    print(f"R2 mediana: {val['r2'].median():.3f} | media: {val['r2'].mean():.3f}")
    print(val[['r2', 'mae', 'rmse', 'mape']].describe().round(2))

    # ========== FIGURA 1: dashboard ==========
    fig = plt.figure(figsize=(18, 15))
    gs  = fig.add_gridspec(4, 3, width_ratios=[1.2, 1.1, 1.1], hspace=0.5, wspace=0.28)
    fig.suptitle(f'{nombre} - diagnostico (validacion)', fontsize=16, fontweight='bold', color=TITULO)

    # (col izq) R2 por CP
    axR = fig.add_subplot(gs[:, 0])
    b = ml_val.sort_values('r2')
    col = [C5 if r < 0.4 else (C3 if r < 0.6 else C1) for r in b['r2']]
    axR.barh(b['cp'], b['r2'], color=col, edgecolor='white')
    axR.axvline(val['r2'].median(), color='gray', lw=1, ls='--')
    axR.set_xlabel('R2'); axR.set_title('R2 por codigo postal', fontsize=11, color=TITULO)
    axR.tick_params(labelsize=8)

    # (col centro) 4 diagnosticos apilados
    axC = fig.add_subplot(gs[0, 1])
    axC.plot(ev['train']['rmse'], label='train', color=C1)
    axC.plot(ev['val']['rmse'], label='val', color=C5)
    axC.set_title('Curva train vs val', fontsize=11, color=TITULO)
    axC.set_xlabel('n arboles'); axC.set_ylabel('RMSE'); axC.legend(fontsize=8)

    axH = fig.add_subplot(gs[1, 1])
    axH.plot(curva.index * 6, curva.values, marker='o', color=C5)
    axH.set_title('Error por horizonte', fontsize=11, color=TITULO)
    axH.set_xlabel('horas'); axH.set_ylabel('MAE')

    axHo = fig.add_subplot(gs[2, 1])
    tt.groupby('hora')['ae'].mean().plot(kind='bar', ax=axHo, color=C1, edgecolor='white')
    axHo.set_title('MAE por hora del dia', fontsize=11, color=TITULO)
    axHo.set_xlabel('hora'); axHo.set_ylabel('MAE'); axHo.tick_params(axis='x', rotation=0)

    axD = fig.add_subplot(gs[3, 1])
    tt.groupby('dia')['ae'].mean().plot(kind='bar', ax=axD, color=C4, edgecolor='white')
    axD.set_title('MAE por dia (0=Lun ... 6=Dom)', fontsize=11, color=TITULO)
    axD.set_xlabel('dia'); axD.set_ylabel('MAE'); axD.tick_params(axis='x', rotation=0)

    # (col der) importancia
    axI = fig.add_subplot(gs[:, 2])
    imp.head(20)[::-1].plot(kind='barh', ax=axI, color=C1, edgecolor='white')
    axI.set_title('Importancia nativa (top 20)', fontsize=11, color=TITULO)
    axI.tick_params(labelsize=8)

    plt.show()

    # ========== FIGURA 2: predicho vs real ==========
    sl = slice(*ventana)
    fig, axes = plt.subplots(len(cps_demo), 1, figsize=(13, 7), sharex=True)
    for axx, cp in zip(np.atleast_1d(axes), cps_demo):
        real = series_val[cp].loc[sl]
        pred = predicciones.loc[predicciones['level'] == cp, 'pred'].sort_index().loc[sl]
        axx.plot(real.index, real.values, color=C1, lw=1.2, label='real')
        axx.plot(pred.index, pred.values, color=C5, lw=1.2, ls='--', label='predicho')
        r2cp = ml_val.loc[ml_val['cp'] == cp, 'r2'].values[0]
        axx.set_title(f'{cp} (R2={r2cp:.2f})', fontsize=10, color=TITULO); axx.set_ylabel('MWh')
    np.atleast_1d(axes)[0].legend(loc='upper right', fontsize=8)
    fig.suptitle(f'{nombre} - predicho vs real ({ventana[0]} a {ventana[1]})', fontweight='bold', color=TITULO)
    plt.tight_layout(); plt.show()

    # ========== FIGURA 3: SHAP ==========
    # X_va ya es todo numerico (la categorica del nivel paso a codigos arriba) -> TreeExplainer sin lios.
    if shap_on:
        Xs = X_va.sample(min(shap_sample, len(X_va)), random_state=12)
        sv = shap.TreeExplainer(reg).shap_values(Xs)
        shap.summary_plot(sv, Xs, max_display=20, show=True)

    return ml_val

In [ ]:
ml_val = revisar_resultados(predicciones, series_val, forecaster, exog_val,
                            nombre='XGBoost por defecto',
                            cps_demo=('08002', '08028', '08039'))

### <font color='#C0392B'><b>2.3 Lectura del modelo por defecto</b></font>

> A completar tras ejecutar (mismo guion de lectura que el 06, para comparar XGBoost vs LightGBM):

 - ¿Manda la autorregresión? Mirar si `lag_1`, `lag_4` (24h) y `lag_28` (semanal) lideran SHAP e importancia, igual que en LightGBM.
 - ¿El barrio importa? Ver si `_level_skforecast` (la identidad del CP) aparece alto -> el modelo global aprende las diferencias entre barrios.
 - Calendario vs clima: peso de `hora`/`dia_semana` frente a `irradiancia`/`temp_mean`/`CDD`.
 - Candidatas a poda: variables que quedan abajo en SHAP (el barrio ya las absorbe).
 - Curva train vs val: ¿se separan las líneas? -> overfitting; ¿dónde se aplana el val? -> margen para tuning.
 - Error por horizonte: debería crecer de 6h a 72h (predecir más lejos cuesta más).
 - Cuándo falla: bloque de mediodía (12:00) y días entre semana suelen ser los más difíciles.
 - Comparar el R² mediana/media con el LightGBM por defecto del 06 (se espera algo muy parecido).

---
# <font color='#1B3A5C'>  **3. Control de overfitting del modelo base (regla José)** </font>

> Punto final de este notebook. Medimos el modelo base a **72h** en dos ventanas comparables y
> calculamos la diferencia relativa de R²:
- **R²_val**: entrena hasta 2024 -> backtest validación 2025 (fuera de muestra real).
- **R²_train**: entrena hasta 2023 -> backtest 2024 (dentro del periodo de train, mismo horizonte).
- **rel_diff** = (R²_train − R²_val) / |R²_train|. Umbral de José: **≤ 0.10** = sin overfitting.

> Ambos R² se miden con el **mismo backtest walk-forward a 72h** -> el rel_diff compara overfitting
> real, no un artefacto de escala. De aquí sale la decisión sobre qué hacer con el modelo.

In [ ]:
# Helper: R2 mediano entre CPs (sobre CPs limpios), desde unas predicciones de backtest
def r2_mediano(pred, series_ref, excluir=()):
    rs = []
    for cp in sorted(pred['level'].unique()):
        if cp in excluir:
            continue
        p = pred.loc[pred['level'] == cp, 'pred'].sort_index()
        y = series_ref[cp].reindex(p.index)
        rs.append(metricas(y.values, p.values)['r2'])
    return float(np.median(rs))

# R2_val: ya lo tenemos -> 'predicciones' de la seccion 2.1 (entrena hasta 2024, backtest 2025)
r2_val = r2_mediano(predicciones, series_val)

# R2_train: entrena hasta 2023 -> backtest 2024 (dentro de train, mismo horizonte 72h)
FIN_2023  = '2023-12-31 18:00:00'
series_tr = {cp: s.loc[:FIN_TRAIN] for cp, s in series.items()}
exog_tr   = {cp: e.loc[:FIN_TRAIN] for cp, e in exog.items()}
idx_2023  = len(series[CPS_TODOS[0]].loc[:FIN_2023])
cv_t = TimeSeriesFold(steps=12, initial_train_size=idx_2023, refit=False)
_, pred_tr = backtesting_forecaster_multiseries(
    forecaster, series=series_tr, exog=exog_tr, cv=cv_t,
    metric='mean_absolute_error', add_aggregated_metric=False, verbose=False, show_progress=False)
r2_train = r2_mediano(pred_tr, series_tr)

rel_diff = (r2_train - r2_val) / abs(r2_train)
print(f"R2_train (2024): {r2_train:.3f}")
print(f"R2_val   (2025): {r2_val:.3f}")
print(f"rel_diff       : {rel_diff:.3f}  ->  "
      f"{'OK, sin overfitting (<=0.10)' if rel_diff <= 0.10 else 'OJO: > 0.10 (overfitting)'}")

elapsed = time.time() - start_time
print(f"\nTiempo de ejecucion: {elapsed/60:.1f} min")

### <font color='#C0392B'><b>3.1 Punto de decisión</b></font>

> Con el R²_val (calidad real fuera de muestra) y el rel_diff (overfitting) del modelo base sobre
> la mesa, aquí se decide el siguiente paso (a rellenar tras ejecutar):

- **rel_diff ≤ 0.10 y R²_val competitivo** -> el modelo por defecto ya generaliza; se puede pasar a tuning fino y al test/multihorizonte.
- **rel_diff > 0.10** -> hay overfitting; toca regularizar (tuning con `reg_alpha`/`reg_lambda`, menos `max_depth`) o podar variables.
- **R²_val por debajo del naive/SARIMA** -> revisar lags y exógenas antes de tunear.
- Comparar R²_val y rel_diff con el LightGBM del 06 para ver cuál parte mejor antes del tuning.